# Prophet — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def make_series(n=60, trend=0.08, amp=2.0, noise=0.25):
    t=np.arange(n)
    return t, 10 + trend*t + amp*np.sin(2*np.pi*t/12) + noise*np.random.randn(n)
def acf(x, max_lag):
    x=np.asarray(x)-np.mean(x); den=np.dot(x,x)
    return np.array([1.0]+[np.dot(x[:-k],x[k:])/den for k in range(1,max_lag+1)])
def moving_average(x,w):
    return np.convolve(x, np.ones(w)/w, mode='valid')
print('setup ok')

## A forecast as trend plus seasonality plus holidays, with changepoints making trend flexible but controlled
Prophet is an additive model with guarded trend flexibility. These examples build the piecewise trend, Fourier seasonality, holiday spike, component sum, and the effect of changepoint regularization.

In [ ]:
# 1) piecewise linear trend changes slope after a changepoint
k=0.5; m=10; delta=-0.3; s=6; t=np.arange(12); trend=k*t+m+delta*np.maximum(0,t-s)
plt.figure(figsize=(6,3)); plt.plot(t,trend,'o-'); plt.axvline(s,color='k',ls='--'); plt.title('slope drops from .5 to .2 after t=6')
assert abs(trend[8]-13.4)<1e-9

In [ ]:
# 2) Fourier terms make smooth seasonality
period=7; t=np.arange(21); season=2*np.sin(2*np.pi*t/period)+1*np.cos(2*np.pi*t/period)
plt.figure(figsize=(6,3)); plt.plot(t,season,'o-'); plt.title('weekly seasonality from sine and cosine')
assert abs(season[3]-(-0.03320138966730257))<1e-12

In [ ]:
# 3) holiday effects are sparse additive spikes
t=np.arange(21); holiday=np.zeros_like(t,dtype=float); holiday[[5,12,19]]=3.0
plt.figure(figsize=(6,3)); plt.stem(t,holiday); plt.title('holiday component is zero except event days')
assert holiday.sum()==9.0

In [ ]:
# 4) full Prophet-style prediction sums components
t=np.arange(14); trend=10+0.3*t; season=np.sin(2*np.pi*t/7); holiday=np.zeros(14); holiday[6]=2; yhat=trend+season+holiday
plt.figure(figsize=(6,3)); plt.plot(t,trend,label='trend'); plt.plot(t,yhat,label='yhat'); plt.legend(); plt.title('yhat = trend + season + holiday')
assert abs(yhat[6]-13.01816851753197)<1e-9

In [ ]:
# 5) changepoint regularization shrinks slope changes
raw=np.array([0.4,-0.3,0.2,-0.1]); strengths=[1.0,0.5,0.0]; paths=[]; t=np.arange(20); cps=[5,9,13,16]
plt.figure(figsize=(6,3))
for lam in strengths:
    d=lam*raw; y=10+0.2*t+sum(di*np.maximum(0,t-c) for di,c in zip(d,cps)); paths.append(y); plt.plot(t,y,label=f'scale={lam}')
plt.legend(); plt.title('smaller flexibility means smoother trend')
assert paths[0][-1]-paths[-1][-1] > 1.0